In [ ]:
# Author: M. Riley Owens (GitHub: mrileyowens)


In [ ]:
import sys

import os
import glob

import h5py

import numpy as np

from astropy.io import fits
import astropy.units as u

import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText

sys.path.append(os.path.abspath('..'))

from mrileyowens.stats import weighted_quantile

In [ ]:
def select():

    '''
    Plot the SFHs of the 2CSFH BEAGLE models
    '''

    # Set common directories
    home = os.getcwd()
    data = f'{home}/data'
    figs = f'{home}/figs'
    results = f'{home}/results'

    files = glob.glob(f'{results}/ew/e24_f775w_dropouts_2csfh_no_lya_ews*.h5')

    hdul = fits.open(f'{data}/JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts.fits')

    # For each BEAGLE fit results file
    for i, file in enumerate(files):

        with h5py.File(file, 'r') as f:

            for j, id in enumerate(list(f.keys())):

                probs = file[id]['probabilities']

                ews_o_iii, ews_h_alpha, ews_h_beta = file[id]['ews_o_iii'], file[id]['ews_h_alpha'], file[id]['ews_h_beta']

                p84_h_alpha = weighted_quantile(ews_h_alpha, probs)
                p84_o_iii_h_beta = weighted_quantile(ews_o_iii + ews_h_beta, probs)

                if p84_h_alpha < 800 and p84_o_iii_h_beta < 800:

                    idx = np.where(hdul[1].data['ID'] == id)

                    f150w = (hdul[1].data['NRC_F150W'] * u.nJy).to(u.ABmag)
                    f200w = (hdul[1].data['NRC_F200W'] * u.nJy).to(u.ABmag)
                    f277w = (hdul[1].data['NRC_F277W'] * u.nJy).to(u.ABmag)

                    color = (f200w - f277w) - (f150w - f200w)

                    if color < 0.3:

                        print('check!')

In [ ]:
select()